# TensorFlow.js Model Fix: Calendar Event Classification

This notebook demonstrates how to build a TensorFlow model that's compatible with TensorFlow.js for calendar event classification. The main fix addresses the "InputLayer should be passed either a batchInputShape or an inputShape" error.

## Key Fixes Applied:
1. **Remove `dtype` parameters** from Input layers (causes TensorFlow.js compatibility issues)
2. **Use proper input shapes** without explicit batch dimensions
3. **Ensure TensorFlow.js export compatibility**

## Model Architecture:
- **Multi-input**: Event name (text), start time, end time
- **Multi-output**: Category classification, Subcategory classification
- **Compatible**: Works with both TensorFlow Python and TensorFlow.js

# 1. Define Categories and Subcategories

First, we define the hierarchical structure of event categories and subcategories that match your application's needs.

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
import json # To save the tokenizer's word_index
import re   # For text preprocessing consistency

print(f"TensorFlow version: {tf.__version__}")

# Define Categories and Subcategories (simplified version for demo)
categories = ['Work', 'Personal', 'Social', 'Health', 'Education', 'Travel']
subcategories = ['Activity', 'Appointment', 'Chore/Errand', 'Class', 'Event', 
                'Gathering', 'Logistics', 'Meeting', 'Other', 'Project', 
                'Social/Family', 'Study Session', 'Test/Exam', 'Trip', 'Work']

# Define mappings
category_map = {cat: i for i, cat in enumerate(categories)}
subcategory_map = {sub: i for i, sub in enumerate(subcategories)}

NUM_CATEGORIES = len(category_map)
NUM_SUBCATEGORIES = len(subcategory_map)

# Define reverse mappings for testing and display
reverse_category_map = {v: k for k, v in category_map.items()}
reverse_subcategory_map = {v: k for k, v in subcategory_map.items()}

print("Defined Categories:", categories)
print("Defined Subcategories:", subcategories)
print(f"Number of categories: {NUM_CATEGORIES}")
print(f"Number of subcategories: {NUM_SUBCATEGORIES}")

# 2. Load and Inspect Data

Load calendar events data from CSV or use hardcoded sample data for demonstration.

In [ ]:
# Load data (using hardcoded sample data for demonstration)
raw_data = [
    {"event_id": "e001", "category": "Health", "subcategory": "Appointment", "name": "Dentist appointment", "start_time": "10:00 AM", "end_time": "11:00 AM"},
    {"event_id": "e002", "category": "Work", "subcategory": "Meeting", "name": "Team sync", "start_time": "09:30 AM", "end_time": "10:00 AM"},
    {"event_id": "e003", "category": "Personal", "subcategory": "Chore/Errand", "name": "Groceries", "start_time": "05:00 PM", "end_time": "06:00 PM"},
    {"event_id": "e004", "category": "Education", "subcategory": "Test/Exam", "name": "Chemistry lab", "start_time": "01:00 PM", "end_time": "02:30 PM"},
    {"event_id": "e005", "category": "Social", "subcategory": "Gathering", "name": "Birthday Celebration", "start_time": "07:00 PM", "end_time": "10:00 PM"},
    {"event_id": "e006", "category": "Health", "subcategory": "Activity", "name": "Gym", "start_time": "06:00 AM", "end_time": "06:45 AM"},
    {"event_id": "e007", "category": "Work", "subcategory": "Project", "name": "Sprint review", "start_time": "03:00 PM", "end_time": "04:00 PM"},
    {"event_id": "e008", "category": "Personal", "subcategory": "Appointment", "name": "Haircut", "start_time": "11:30 AM", "end_time": "12:00 PM"},
    {"event_id": "e009", "category": "Social", "subcategory": "Gathering", "name": "Book club meeting", "start_time": "08:00 PM", "end_time": "09:00 PM"},
    {"event_id": "e010", "category": "Education", "subcategory": "Study Session", "name": "Study for Math", "start_time": "02:00 PM", "end_time": "04:00 PM"},
    {"event_id": "e011", "category": "Travel", "subcategory": "Trip", "name": "Flight to London", "start_time": "10:00 AM", "end_time": "04:00 PM"},
    {"event_id": "e012", "category": "Travel", "subcategory": "Logistics", "name": "Airport drop-off", "start_time": "06:00 AM", "end_time": "06:30 AM"},
    {"event_id": "e013", "category": "Personal", "subcategory": "Chore/Errand", "name": "Bank", "start_time": "12:00 PM", "end_time": "12:30 PM"},
    {"event_id": "e014", "category": "Social", "subcategory": "Gathering", "name": "Dinner with friends", "start_time": "07:30 PM", "end_time": "09:00 PM"},
    {"event_id": "e015", "category": "Health", "subcategory": "Activity", "name": "Yoga class", "start_time": "07:00 AM", "end_time": "08:00 AM"},
]

df = pd.DataFrame(raw_data).fillna('')

print(f"Loaded {len(df)} data points")
print("DataFrame Columns:", df.columns.tolist())
print("\nFirst few rows:")
print(df.head())

# 3. Text Preprocessing and Tokenizer Setup

Set up text preprocessing and tokenization for event names.

In [ ]:
# Constants
MAX_SEQUENCE_LENGTH = 15  # Maximum length for text sequences

# Build vocabulary from event names
all_names = df['name'].tolist()

tokenizer = Tokenizer(num_words=None, oov_token="<unk>")
tokenizer.fit_on_texts(all_names)

# Save the word_index for use in JavaScript frontend
word_index_path = 'word_index.json'
with open(word_index_path, 'w') as f:
    json.dump(tokenizer.word_index, f)
print(f"Word index saved to {word_index_path}")

# Get vocabulary size for embedding layer
VOCAB_SIZE = len(tokenizer.word_index) + 1  # +1 for padding/OOV token

print(f"Vocabulary size: {VOCAB_SIZE}")
print(f"Sample words from vocabulary: {list(tokenizer.word_index.items())[:10]}")

# 4. Feature Engineering: Time and Text

Define preprocessing functions for text and time features.

In [ ]:
def preprocess_text(text, tokenizer_obj, max_length):
    """Tokenizes, indexes using the fitted tokenizer, and pads/truncates text."""
    if pd.isna(text) or text == "":
        text = ""
    
    sequences = tokenizer_obj.texts_to_sequences([text])
    padded = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')
    return padded[0].tolist()

def preprocess_time(time_string):
    """Converts time string to numerical features (normalized hour, AM/PM indicators)."""
    if pd.isna(time_string) or time_string == "":
        return [0, 0, 0, 0]  # Default values if time is missing

    try:
        if 'AM' in time_string.upper() or 'PM' in time_string.upper():
            dt_obj = pd.to_datetime(time_string, format='%I:%M %p', errors='coerce')
        elif ':' in time_string:
            dt_obj = pd.to_datetime(time_string, format='%H:%M', errors='coerce')
        else:
            dt_obj = pd.to_datetime(time_string, format='%H', errors='coerce')

        if pd.isna(dt_obj):
            raise ValueError(f"Could not parse time string: {time_string}")

        hour = dt_obj.hour  # 0-23
        is_morning = 1 if (hour >= 6 and hour < 12) else 0
        is_afternoon = 1 if (hour >= 12 and hour < 18) else 0
        is_evening = 1 if (hour >= 18 or hour < 6) else 0

        normalized_hour = hour / 23.0

        return [normalized_hour, is_morning, is_afternoon, is_evening]
    except Exception as e:
        return [0, 0, 0, 0]

# Test the functions
print("Testing text preprocessing:")
sample_text = "Team sync meeting"
print(f"Original: '{sample_text}'")
print(f"Processed: {preprocess_text(sample_text, tokenizer, MAX_SEQUENCE_LENGTH)}")

print("\nTesting time preprocessing:")
sample_time = "09:30 AM"
print(f"Original: '{sample_time}'")
print(f"Processed: {preprocess_time(sample_time)}")

# 5. Prepare Model Inputs and Outputs

Convert preprocessed features and labels into NumPy arrays ready for model training.

## 🔧 Model Architecture - TensorFlow.js Compatible

The key to fixing the "InputLayer should be passed either a batchInputShape or an inputShape" error is to define Input layers WITHOUT dtype parameters and ensure proper shape specifications for TensorFlow.js compatibility.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# CRITICAL: Define Input layers WITHOUT dtype parameter for TensorFlow.js compatibility
# This fixes the "InputLayer should be passed either a batchInputShape or an inputShape" error

# Text input - tokenized sequences
text_input = Input(shape=(100,), name='text_input')  # Remove dtype=tf.int32

# Time input - hour as float
time_input = Input(shape=(1,), name='time_input')    # Remove dtype=tf.float32

# Duration input - duration in hours
duration_input = Input(shape=(1,), name='duration_input')  # Remove dtype=tf.float32

print("✅ Input layers defined without dtype (TensorFlow.js compatible)")
print(f"Text input shape: {text_input.shape}")
print(f"Time input shape: {time_input.shape}")
print(f"Duration input shape: {duration_input.shape}")

In [ ]:
# Text processing branch
embedding = Embedding(input_dim=5000, output_dim=128, input_length=100)(text_input)
lstm = LSTM(64, dropout=0.2, recurrent_dropout=0.2)(embedding)

# Combine all features
combined = Concatenate()([lstm, time_input, duration_input])

# Hidden layers
hidden1 = Dense(128, activation='relu')(combined)
dropout1 = Dropout(0.3)(hidden1)
hidden2 = Dense(64, activation='relu')(dropout1)
dropout2 = Dropout(0.2)(hidden2)

# Output layers - Multi-output model
category_output = Dense(len(categories), activation='softmax', name='category_output')(dropout2)
subcategory_output = Dense(len(subcategories), activation='softmax', name='subcategory_output')(dropout2)

# Create the model
model = Model(
    inputs=[text_input, time_input, duration_input],
    outputs=[category_output, subcategory_output],
    name='event_classifier'
)

print("✅ Model architecture created successfully!")
model.summary()

## 📊 Model Compilation

Configure the model with appropriate loss functions and metrics for multi-output classification.

In [ ]:
# Compile the model with appropriate loss functions for multi-output
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss={
        'category_output': 'sparse_categorical_crossentropy',
        'subcategory_output': 'sparse_categorical_crossentropy'
    },
    metrics={
        'category_output': ['accuracy'],
        'subcategory_output': ['accuracy']
    },
    loss_weights={
        'category_output': 1.0,
        'subcategory_output': 0.8  # Slightly less weight on subcategory
    }
)

print("✅ Model compiled successfully!")
print("\nModel inputs:")
for i, input_layer in enumerate(model.inputs):
    print(f"  {i}: {input_layer.name} - {input_layer.shape}")
    
print("\nModel outputs:")
for i, output_layer in enumerate(model.outputs):
    print(f"  {i}: {output_layer.name} - {output_layer.shape}")

## 🎯 Model Training (Sample)

For demonstration purposes, we'll create some sample training data. In practice, you would use your actual dataset.

## Export and Convert Model for TensorFlow.js

After training your model, follow these steps to export and convert it for use in your Next.js app:

1. **Export the model to the SavedModel format:**
   ```python
   model.save('exported_model')
   ```
2. **Install the TensorFlow.js converter (if not already installed):**
   ```bash
   pip install tensorflowjs
   ```
3. **Convert the SavedModel to TensorFlow.js format:**
   ```bash
   tensorflowjs_converter --input_format=tf_saved_model --output_format=tfjs_graph_model exported_model ./tfjs_model
   ```
4. **Copy the contents of `./tfjs_model` (model.json and shard files) to your Next.js app's `public/` directory.**

5. **Test the integration in your app.**

If you encounter any issues, ensure your Keras Input layers are defined with only the `shape` argument (no `dtype`).

In [ ]:
# Generate some random sample data for demonstration
num_samples = 100
X_text = np.random.randint(1, 5000, size=(num_samples, 100))
X_time = np.random.rand(num_samples, 1)
X_duration = np.random.rand(num_samples, 1)
y_category = np.random.randint(0, len(categories), size=(num_samples, 1))
y_subcategory = np.random.randint(0, len(subcategories), size=(num_samples, 1))

# Train the model (for demonstration, use very few epochs)
history = model.fit(
    [X_text, X_time, X_duration],
    {'category_output': y_category, 'subcategory_output': y_subcategory},
    epochs=2,
    batch_size=8,
    validation_split=0.2
)

print("✅ Model trained (demo)")